In [41]:
# *** what we will do ***

In [6]:
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "Data"
RAW_DIR = DATA_DIR / "Dataset"
WAREHOUSE_DIR = DATA_DIR / "Warehouse"
csv = RAW_DIR / "olist_orders_dataset.csv"

In [4]:
print(duckdb.sql(f"SELECT count(*) FROM '{csv}'"))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        99441 │
└──────────────┘



In [17]:
DB_PATH = WAREHOUSE_DIR / "ecommerce.duckdb"
con = duckdb.connect(str(DB_PATH))

In [18]:
def q(sql_str: str) -> pd.DataFrame:
    """Run a SQl query in duckdv and return a pandas dataframe."""
    return con.sql(sql_str).df()

In [46]:
### 2. Load the source tables into a reaw schema

In [19]:
RAW_TABLES: dict[ str, dict[str,str]]= {
    "customers": {"customer_zip_code_perfix": "VARCHAR"}, # KEEP LEADING ZEROS
    "orders": {},
    "order_items": {},
    "products": {},
    "sellers": {}
    
}

RAW_SCHEMA = "raw"

In [10]:
def csv_path(table: str) -> Path:
    return RAW_DIR / f"olist_{table}_dataset.csv"

def check_source_files(table) -> None:
    missing = {csv_path(t) for t in table if not csv_path(t).exists()}
    if missing: 
        names = "\n".joint(f" - {p}" for p in missing)
        raise FileNotFoundError(f"missing CSV files: {names}")

In [20]:
def load_raw_tables(con: duckdb.DuckDBPyConnection, table: str, column_types: dict[str, str]) -> int:
    target = f'{RAW_SCHEMA}."{table}"'
    source = con.read_csv(str(csv_path(table)), header=True, dtype=column_types or None)
    con.execute(f"Drop TABLEIF EXISTS {target}")
    source.to_table(target)
    return con.execute(f"SELECT count(*) FROM {target}").fetchone()[0]

In [12]:
check_source_files(RAW_TABLES)

In [22]:
con.execute(f"CREATE SCHEMA IF NOT EXISTS {RAW_SCHEMA}")
row_counts = {t: load_raw_tables(con, t, types) for t, types in RAW_TABLES.items()}

BinderException: Binder Error: COLUMN_TYPES error: Columns with names: "customer_zip_code_perfix" do not exist in the CSV File